In [3]:
import geopandas as gpd
import pandas as pd
import xarray as xr
import sys
import geemap 
from tqdm.notebook import tqdm
import glob 
import sys
import importlib

path_function = '../../Hugo/a_b_c_functions/*'

path_function = glob.glob(path_function)
# redirection chemin fonction
for lib in path_function:
    # print(lib)
    sys.path.insert(1,lib)

# Outils pour les opérations spatiales (from hugo)
from utils_vector import gdf_to_bbox
# from utils_raster import *
from get_boundary import get_boundary # mini utilitaire permettant de définir des aoi rapidement (ex: une ville)
from utils_proj import get_utm_epsg
from utils_date import *
from utils_gee import * 
from utils_gee import prepare_ds_xarray_ee
from utils_raster import polygon_to_raster, raster_to_polygon
import warnings

service_account = 'gee-141@gee161025.iam.gserviceaccount.com'
credentials = ee.ServiceAccountCredentials(service_account, 'gee161025-533af22f806b.json')
ee.Initialize(credentials)

In [4]:
import requests

# ── AOI : CA La Roche-sur-Yon - Agglomération ────────────────────────────────
url_epci = "https://geo.api.gouv.fr/epcis?code=248500589&format=geojson&geometry=contour"
aoi = gpd.read_file(url_epci)

print(f"✓ AOI : {aoi.iloc[0]['nom']}")
print(f"  Population : {aoi.iloc[0]['population']:,}")

# CRS de travail
utm_epsg = f"EPSG:{get_utm_epsg(gdf=aoi)}" # utm_epsg est le crs à adopter car le plus précis
aoi = aoi.to_crs(utm_epsg)
aoi_ee = geemap.gdf_to_ee(aoi)

print(f"  Surface : {aoi.geometry.area.sum() / 1e6:.1f} km²")
print(f"  CRS : {utm_epsg}")
aoi.explore()

Skipping field codesDepartements: unsupported OGR type: 5
Skipping field codesRegions: unsupported OGR type: 5


✓ AOI : CA La Roche-sur-Yon - Agglomération
  Population : 98,488
  Surface : 502.2 km²
  CRS : EPSG:32630


In [9]:
# ── Extraction WorldCover 2021 sur l'agglomération ─────────────────
wc_2021 = ee.ImageCollection('ESA/WorldCover/v200').mosaic()

wc_2021 = ee.ImageCollection('ESA/WorldCover/v200').mosaic().clip(aoi_ee) # Le .clip(aoi_ee) force les pixels hors polygone à devenir "NoData"
wc_2021 = prepare_ds_xarray_ee(
    wc_2021,
    scale=10,
    geometry=aoi_ee,
    crs=utm_epsg
).compute()

# Transfo Dataset → DataArray
# to dig and fully understand
wc_2021 = wc_2021['Map']

print(f"  Dimensions : {wc_2021.dims}")
print(f"  Shape : {wc_2021.shape}")
import numpy as np
valeurs = np.unique(wc_2021.values)
valeurs = valeurs[valeurs > 0]  # exclure nodata
print(f"  Valeurs uniques ({len(valeurs)}) : {valeurs.astype(int).tolist()}")

# wc_2021_poly = raster_to_polygon(wc_2021, data_type = 'uint8')
# wc_2021_poly.explore()

/home/jovyan/.local/lib/python3.11/site-packages/xee/ext.py:696: UserWarning: Unable to retrieve 'system:time_start' values from an ImageCollection due to: No 'system:time_start' values found in the 'ImageCollection'.
  warnings.warn(


  Dimensions : ('date', 'y', 'x')
  Shape : (1, 3105, 3126)
  Valeurs uniques (7) : [10, 30, 40, 50, 60, 80, 90]


In [6]:
from utils_vector import unify_data

# Chargement des dalles CosIA 
cosia_path_pattern = "/home/jovyan/nfs/team/marion/data/cosia/D085_*.gpkg"

dalles = glob.glob(cosia_path_pattern)
print(f"✓ {len(dalles)} dalle(s) trouvée(s)")

# Chargement
cosia_raw = unify_data(
    path_pattern=cosia_path_pattern,
    source_col='dalle_source',
    force_lowercase=True
)

print(f"  Shape : {cosia_raw.shape}")
print(f"  CRS natif : {cosia_raw.crs}")
print(f"  Colonnes : {cosia_raw.columns.tolist()}")

✓ 20 dalle(s) trouvée(s)
succesfully unified: D085_2022_370_6650_vecto.gpkg
succesfully unified: D085_2022_360_6650_vecto.gpkg
succesfully unified: D085_2022_350_6650_vecto.gpkg
succesfully unified: D085_2022_340_6650_vecto.gpkg
succesfully unified: D085_2022_340_6630_vecto.gpkg
succesfully unified: D085_2022_350_6630_vecto.gpkg
succesfully unified: D085_2022_360_6630_vecto.gpkg
succesfully unified: D085_2022_370_6630_vecto.gpkg
succesfully unified: D085_2022_350_6620_vecto.gpkg
succesfully unified: D085_2022_340_6620_vecto.gpkg
succesfully unified: D085_2022_370_6620_vecto.gpkg
succesfully unified: D085_2022_360_6620_vecto.gpkg
succesfully unified: D085_2022_340_6660_vecto.gpkg
succesfully unified: D085_2022_360_6640_vecto.gpkg
succesfully unified: D085_2022_350_6660_vecto.gpkg
succesfully unified: D085_2022_370_6640_vecto.gpkg
succesfully unified: D085_2022_340_6640_vecto.gpkg
succesfully unified: D085_2022_360_6660_vecto.gpkg
succesfully unified: D085_2022_350_6640_vecto.gpkg
succes

In [7]:
# ── Clip CosIA sur l'agglomération ─────────────────────────────────
aoi_l93 = aoi.to_crs(epsg=2154)

cosia_clipped = gpd.clip(cosia_raw, aoi_l93)
cosia_clipped = cosia_clipped.to_crs(utm_epsg)

print(f"✓ CosIA clippé")
print(f"  Polygones : {len(cosia_clipped)}")
print(f"  Classes présentes : {sorted(cosia_clipped['classe'].unique().tolist())}")

✓ CosIA clippé
  Polygones : 584195
  Classes présentes : ['Broussaille', 'Bâtiment', 'Conifère', 'Culture', 'Feuillu', 'Pelouse', 'Piscine', 'Serre', 'Sol nu', 'Surface eau', 'Terre labourée', 'Vigne', 'Zone imperméable', 'Zone perméable']


In [16]:
import osmnx as ox
import geopandas as gpd

# 1. Récupération des routes via l'AOI
# On utilise l'AOI en EPSG:4326 car OSM travaille en coordonnées géographiques
aoi_4326 = aoi.to_crs(epsg=4326)

print("  Téléchargement des routes OSM...")
# "drive" récupère toutes les routes circulables
highways = ox.features_from_polygon(aoi_4326.geometry.unary_union, tags={"highway": True})

# 2. Nettoyage et Projection
# On ne garde que les colonnes utiles et on repasse dans ton CRS de travail (UTM)
highways = highways[highways.geometry.type.isin(['LineString', 'MultiLineString'])]
highways = highways.to_crs(utm_epsg)

# 3. Bufferisation : transformer les lignes en surfaces
# On définit des largeurs moyennes selon le type de route
def get_buffer_width(highway_type):
    if highway_type in ['motorway', 'trunk']: return 15
    if highway_type in ['primary', 'secondary']: return 10
    if highway_type in ['tertiary', 'residential']: return 6
    return 4 # petites routes, chemins

highways['width'] = highways['highway'].apply(get_buffer_width)
highways['geometry'] = highways.geometry.buffer(highways['width'] / 2)

# 4. Préparation pour la fusion avec CosIA
highways_to_join = highways[['geometry']].copy()
highways_to_join['classe'] = 'Highway'
highways_to_join['wc_code'] = 51  # Nouvelle classe dédiée : Highway

print(f"  {len(highways_to_join)} segments de routes convertis en polygones.")

  Téléchargement des routes OSM...


/tmp/ipykernel_115/1697202761.py:10: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  highways = ox.features_from_polygon(aoi_4326.geometry.unary_union, tags={"highway": True})


  21779 segments de routes convertis en polygones.


In [18]:
# 1. Définition du dictionnaire de mapping pour CosIA
mapping_to_wc = {
    'Bâtiment': 50,
    'Zone imperméable': 50,
    'Zone perméable': 60,
    'Sol nu': 60,
    'Surface eau': 80,
    'Piscine': 80,
    'Conifère': 10,
    'Feuillu': 10,
    'Broussaille': 20,
    'Pelouse': 30,
    'Serre': 40,
    'Culture': 40,
    'Terre labourée': 40,
    'Vigne': 40,
    'Highway': 51 # Ajout de la classe OSM
}

# 2. Application du mapping sur CosIA
cosia_clipped['wc_code'] = cosia_clipped['classe'].map(mapping_to_wc)

# 3. INITIALISATION du dictionnaire de priorités
# Plus le chiffre est élevé, plus la classe est "au-dessus" des autres
priorites = {
    60: 1,   # Sol nu
    50: 2,  # Bâtiment
    40: 3,   # Cultures
    30: 4,   # Herbe
    20: 5,   # Arbustes
    10: 6,   # Arbres
    80: 7,  # Eau
    51: 8  # Routes OSM (Priorité maximale)
}

# 4. Fusion CosIA + OSM
cosia_final = pd.concat([cosia_clipped, highways_to_join], ignore_index=True)

# 5. Rasterisation
cosia_raster = polygon_to_raster(
    shapefile=cosia_final,
    img_ref=wc_2021,
    column='wc_code',
    priority_list=priorites,
    fill_value=0
)

In [ ]:
# On rasterise UNIQUEMENT les routes OSM pour créer le masque
osm_mask_raster = polygon_to_raster(
    shapefile=highways_to_join,
    img_ref=wc_2021,
    column='wc_code',
    fill_value=0
)

# On crée la version augmentée de WorldCover
# Si le pixel est une route (51) dans OSM, on met 51, sinon on garde l'original
wc_2021_augmented = xr.where(osm_mask_raster == 51, 51, wc_2021)

print("✓ WorldCover augmenté avec la classe Highway (51)")

print(f"Valeurs dans CosIA : {np.unique(cosia_raster.values)}")
print(f"Valeurs dans WC Augmenté : {np.unique(wc_2021_augmented.values)}")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))

# Affichage simple des codes
cosia_raster.plot(ax=ax1, add_colorbar=False)
ax1.set_title("CosIA + OSM")

wc_2021_augmented.plot(ax=ax2, add_colorbar=False)
ax2.set_title("WorldCover + OSM")

plt.show()